# 02 · Random walks for GraphNLI

For every toot, one weighted random walk of length 4 over its conversation tree
(the walk strategy of GraphNLI): from the current toot, the walk moves to the
parent with weight 0.6 and splits the remaining 0.4 equally between the children,
using only toots observed on the same instance. A walk that cannot move further
stops early and its remaining positions stay empty.

Each row of the output CSV is one training sample: the target toot (`sent1`), the
three context toots of the walk (`sent2`–`sent4`) and the target's toxicity score.
This CSV is the `--data_csv` input of `2_FL_GraphNLI_model_training/train.py`.

In [ ]:
import csv
import pickle
import random
from pathlib import Path

import pandas as pd

In [ ]:
DATASET = "pleroma"  # or "mastodon"

GRAPHNLI_DIR = Path("../../data") / DATASET / "graphnli"
INPUT_PKL = GRAPHNLI_DIR / f"{DATASET}_graphnli_conversations.pkl"
OUTPUT_CSV = GRAPHNLI_DIR / f"{DATASET}_graphnli_random_walks.csv"

WALK_LENGTH = 4
PARENT_WEIGHT = 0.6
RANDOM_SEED = 42

In [ ]:
def weighted_random_walk(sentences, data, node_id, child_edges, walk_len, instance, weight=PARENT_WEIGHT):
    """Fill ``sentences`` with the texts along one walk; returns (sentences, toxicity of the start toot)."""
    sentences[0] = data["nodes"][node_id]["text"]
    toxicity = data["nodes"][node_id]["toxicity"]

    for i in range(1, walk_len):
        choices, probs = [], []

        parent = data["edges"].get(node_id, "")
        if parent != "" and instance in data["nodes"][parent]["old_conv_ids"]:
            choices.append(parent)
            probs.append(weight)

        children = child_edges.get(node_id, [])
        for child in children:
            if instance in data["nodes"][child]["old_conv_ids"]:
                choices.append(child)
                probs.append((1 - weight) / len(children))

        if not choices:
            return sentences, toxicity

        node_id = random.choices(choices, probs)[0]
        sentences[i] = data["nodes"][node_id]["text"]

    return sentences, toxicity


def child_lookup(data):
    """parent -> children, from ``data["edges"][child] = parent``."""
    children = {}
    for node_id in data["nodes"]:
        parent = data["edges"].get(node_id, "")
        if parent != "":
            children.setdefault(parent, []).append(node_id)
    return children


def read_stream(path):
    with open(path, "rb") as f:
        while True:
            try:
                yield pickle.load(f)
            except EOFError:
                return

In [ ]:
random.seed(RANDOM_SEED)

fields = ["instance_toot_found_id", "conversation_id", "toot_id", "sent1", "sent2", "sent3", "sent4", "toxicity"]
n_rows = 0

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=fields)
    writer.writeheader()

    for record in read_stream(INPUT_PKL):
        inst = str(record["instance_toot_found_id"])
        data = record["graph"]
        children = child_lookup(data)

        for node_id in data["nodes"]:
            sentences, toxicity = weighted_random_walk([""] * WALK_LENGTH, data, node_id, children, WALK_LENGTH, inst)
            writer.writerow({
                "instance_toot_found_id": inst,
                "conversation_id": str(record["conversation_id"]),
                "toot_id": str(node_id),
                "sent1": sentences[0],
                "sent2": sentences[1],
                "sent3": sentences[2],
                "sent4": sentences[3],
                "toxicity": toxicity,
            })
            n_rows += 1

print(f"{n_rows:,} walks → {OUTPUT_CSV}")

In [ ]:
pd.read_csv(OUTPUT_CSV, nrows=5)